In [8]:
from cProfile import label

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,log_loss,classification_report
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

# Cell 2: 加载经典鸢尾花数据集（三分类）
from sklearn.datasets import load_iris

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target  # 0, 1, 2 三类

print("特征形状:", X.shape)
print("标签分布:", np.bincount(y))
X.head()

特征形状: (150, 4)
标签分布: [50 50 50]


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [9]:
# Cell 3: 拆分数据 + 训练 XGBoost 多分类模型
# 1. 划分训练集/测试集（这里先用随机划分学 API）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. 初始化 XGBoost 分类器
# multi:softprob 是多分类概率输出，num_class 指定类别数
model = XGBClassifier(
    n_estimators=100,          # 树的数量
    max_depth=4,               # 单棵树最大深度（防止过拟合）
    learning_rate=0.1,         # 学习率（每棵树的贡献）
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',    # 多分类对数损失
    random_state=42
)

# 3. 训练模型（eval_set 用于观察训练过程）
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False # 设为 True 可以看到每轮的评估指标
)

# 4. 预测
y_pred = model.predict(X_test)          # 预测类别
y_pred_proba = model.predict_proba(X_test)  # 预测概率（5类概率分布）

print("测试集准确率:", accuracy_score(y_test, y_pred))
print("Top1 预测概率示例:", y_pred_proba[0].round(3))  # 三个类别的概率
print(y_pred)

测试集准确率: 0.9333333333333333
Top1 预测概率示例: [0.991 0.007 0.002]
[0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 1 0 2 1 2 2 2 1 0 2 0]


In [10]:

# 22维特征+标签，完全对齐你的AI预测模块需求
df = pd.DataFrame({
    # ==================== 车手状态维度（6维）====================
    "qualifying_position": [1, 2, 3, 4, 5, 1, 3, 1, 2, 12],  # 排位赛位次（核心特征）
    "recent_5_race_avg_pos": [1.2, 2.8, 3.1, 4.5, 5.2, 1.1, 3.0, 1.3, 2.7, np.nan],  # 近5场平均完赛位次（新秀为NaN）
    "career_podium_rate": [0.45, 0.32, 0.28, 0.15, 0.08, 0.46, 0.30, 0.47, 0.33, 0.0],  # 职业生涯领奖台率
    "wet_weather_performance": [0.8, 0.6, 0.7, 0.5, 0.4, 0.82, 0.58, 0.81, 0.61, 0.3],  # 雨战能力（0-1评分）
    "overtaking_success_rate": [0.72, 0.65, 0.58, 0.42, 0.35, 0.75, 0.62, 0.73, 0.66, 0.2],  # 超车成功率
    "is_rookie": [0, 0, 0, 0, 1, 0, 0, 0, 0, 1],  # 是否为新秀（1=是，0=否）

    # ==================== 车队趋势维度（5维）====================
    "constructor_recent_3_race_points": [120, 98, 85, 62, 41, 118, 92, 125, 102, 15],  # 车队近3场总积分
    "car_performance_index": [0.92, 0.88, 0.85, 0.78, 0.72, 0.93, 0.86, 0.94, 0.89, 0.65],  # 赛车性能指数（0-1）
    "pit_stop_avg_time": [2.1, 2.2, 2.3, 2.4, 2.5, 2.0, 2.3, 2.1, 2.2, 2.8],  # 平均进站时间（秒）
    "reliability_rate": [0.95, 0.92, 0.89, 0.85, 0.78, 0.96, 0.91, 0.96, 0.93, 0.7],  # 赛车可靠性（完赛率）
    "team_radio_efficiency": [0.88, 0.82, 0.79, 0.75, 0.68, 0.89, 0.80, 0.90, 0.83, 0.6],  # 车队沟通效率（0-1）

    # ==================== 赛道特性维度（6维）====================
    "track_historical_best_pos": [1, 2, 3, 4, 5, 1, 3, 1, 2, 12],  # 车手该赛道历史最佳位次
    "track_type": [0, 0, 0, 0, 0, 1, 1, 0, 0, 0],  # 赛道类型：0=高速，1=街道，2=技术型
    "overtaking_difficulty": [0.3, 0.4, 0.6, 0.7, 0.8, 0.5, 0.6, 0.3, 0.4, 0.9],  # 超车难度（0-1，越高越难超）
    "tire_degradation_rate": [0.05, 0.06, 0.07, 0.08, 0.09, 0.06, 0.07, 0.05, 0.06, 0.1],  # 轮胎衰减率（%/圈）
    "drs_zone_length_ratio": [0.25, 0.25, 0.22, 0.20, 0.18, 0.24, 0.24, np.nan, np.nan, np.nan],  # DRS区域占比（2026年为空）
    "track_evolution_rate": [0.03, 0.03, 0.04, 0.04, 0.05, 0.03, 0.03, 0.03, 0.03, 0.06],  # 赛道抓地力进化速率

    # ==================== 环境维度（5维）====================
    "air_temp": [28, 29, 30, 31, 32, 26, 27, 29, 30, 31],  # 空气温度（℃）
    "track_temp": [42, 43, 44, 45, 46, 40, 41, 43, 44, 45],  # 赛道温度（℃）
    "rain_probability": [0.1, 0.15, 0.2, 0.25, 0.3, 0.05, 0.1, 0.12, 0.18, 0.25],  # 降雨概率
    "wind_speed": [5, 6, 7, 8, 9, 4, 5, 5, 6, 7],  # 风速（km/h）
    "humidity": [45, 48, 50, 52, 55, 42, 44, 46, 49, 52],  # 湿度（%）

    # ==================== 标签（多分类，5类）====================
    # 0=Top1, 1=Top2-3, 2=Top4-5, 3=Top6-10, 4=其他
    "label": [0, 1, 1, 2, 3, 0, 1, 0, 1, 4],

    # ==================== 辅助字段（非特征，用于数据管理）====================
    "year": [2024, 2024, 2024, 2024, 2024, 2024, 2024, 2026, 2026, 2026],
    "round": [1, 1, 1, 1, 1, 2, 2, 1, 1, 1],
    "driver_code": ["VER", "HAM", "LEC", "NOR", "PIA", "VER", "HAM", "VER", "HAM", "ROG"]
})



F1 特征维度: (10, 22)
标签分布: label
0    3
1    4
2    1
3    1
4    1
Name: count, dtype: int64
训练样本: 7, 测试样本: 3


In [12]:
# Cell 4: 用之前给你的 F1 特征数据训练
# 1. 分离特征与标签
y = df["label"]  # 0=Top1, 1=Top3, 2=Top5, 3=Top10, 4=其他
X = df.drop(columns=["label", "year", "round", "driver_code"])

print("F1 特征维度:", X.shape)
print("标签分布:", y.value_counts().sort_index())

# 2. ⚠️ 关键：时间序列划分，不能用随机划分！
# 用 2024 及以前做训练，2026 做测试，避免"未来数据泄露"
train_mask = df["year"] < 2025
test_mask = df["year"] == 2026

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"训练样本: {len(X_train)}, 测试样本: {len(X_test)}")

# 3. 训练 F1 预测模型
f1_model = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=5,
    eval_metric='mlogloss',
    random_state=42
)

f1_model.fit(X_train, y_train)

# 4. 评估
y_pred = f1_model.predict(X_test)
y_pred_proba = f1_model.predict_proba(X_test)

print("测试集准确率:", accuracy_score(y_test, y_pred))
print("LogLoss:", log_loss(y_test, y_pred_proba))
print("\n分类报告:")
print(classification_report(y_test, y_pred))

F1 特征维度: (10, 22)
标签分布: label
0    3
1    4
2    1
3    1
4    1
Name: count, dtype: int64
训练样本: 7, 测试样本: 3
测试集准确率: 0.3333333333333333


ValueError: y_true and y_prob contain different number of classes: 3 vs 4. Please provide the true labels explicitly through the labels argument. Classes found in y_true: [0 1 4]